<a href="https://colab.research.google.com/github/smstrong920/GB885-Final-Project---Strong---S/blob/main/GB885_Final_Project_Analysis_Strong_S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GB885 Final Project Analysis
Spencer Strong

#####In this notebook I will upload the cleaned files created in the ingestion/cleaning notebook and do analysis on them

In [1]:
#import needed modules
import pandas as pd
from google.colab import files

In [2]:
#All three of the parquet cleaned data files must be uploaded here
uploaded = files.upload()

df_product_and_sales = pd.read_parquet('/content/df_product_and_sales.parquet')
df_state = pd.read_parquet('/content/df_state.parquet')
df_retailer = pd.read_parquet('/content/df_retailer.parquet')

Saving df_retailer.parquet to df_retailer.parquet
Saving df_state.parquet to df_state.parquet
Saving df_product_and_sales.parquet to df_product_and_sales.parquet


In [3]:
#verify files were uploaded
print(df_product_and_sales.head())
print(df_state.head())
print(df_retailer.head())
#upload was succeful

  ORDER_ID RETAILER_ID INVOICE_DATE  MONTH  DAY  YEAR PRODUCT_ID  \
0        1    A00MOHCO   2020-01-01      1    1  2020         20   
1        7    A00MOHCO   2020-01-07      1    7  2020         20   
2       13    A00MOHCO   2020-01-25      1   25  2020         20   
3       19    A00MOHCO   2020-01-31      1   31  2020         20   
4       25    A00MOHCO   2020-02-06      2    6  2020         20   

   PRICE_PER_UNIT  UNITS_SOLD  OPERATING_MARGIN SALES_METHOD  \
0            50.0        1200               0.5     In-store   
1            50.0        1250               0.5     In-store   
2            50.0        1220               0.5       Outlet   
3            50.0        1200               0.5       Outlet   
4            60.0        1220               0.5       Outlet   

            PRODUCT_NAME  
0  Men's Street Footwear  
1  Men's Street Footwear  
2  Men's Street Footwear  
3  Men's Street Footwear  
4  Men's Street Footwear  
  ORDER_ID RETAILER_ID INVOICE_DATE  MONTH  

In [4]:
#ensure proper datatypes were maintained
df_product_and_sales.info()
df_state.info()
df_retailer.info()
#datatypes look good let's rock and roll!

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDER_ID          9648 non-null   object        
 1   RETAILER_ID       9648 non-null   object        
 2   INVOICE_DATE      9648 non-null   datetime64[ns]
 3   MONTH             9648 non-null   int64         
 4   DAY               9648 non-null   int64         
 5   YEAR              9648 non-null   int64         
 6   PRODUCT_ID        9648 non-null   object        
 7   PRICE_PER_UNIT    9648 non-null   float64       
 8   UNITS_SOLD        9648 non-null   int64         
 9   OPERATING_MARGIN  9648 non-null   float64       
 10  SALES_METHOD      9648 non-null   object        
 11  PRODUCT_NAME      9648 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(5)
memory usage: 904.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9591

In [5]:
# Create the dollar sales column on all three frames
for df in [df_product_and_sales, df_state, df_retailer]:
    df['TOTAL_SALES'] = df['PRICE_PER_UNIT'] * df['UNITS_SOLD']

In [6]:
#verify
df_product_and_sales[['PRICE_PER_UNIT', 'UNITS_SOLD', 'TOTAL_SALES']].head()

,PRICE_PER_UNIT,UNITS_SOLD,TOTAL_SALES
0,50.0,1200,60000.0
1,50.0,1250,62500.0
2,50.0,1220,61000.0
3,50.0,1200,60000.0
4,60.0,1220,73200.0


In [7]:
#veriy
df_state[['PRICE_PER_UNIT', 'UNITS_SOLD', 'TOTAL_SALES']].head()

,PRICE_PER_UNIT,UNITS_SOLD,TOTAL_SALES
0,50.0,1200,60000.0
1,50.0,1250,62500.0
2,50.0,1220,61000.0
3,50.0,1200,60000.0
4,60.0,1220,73200.0


In [8]:
#verify
df_retailer[['PRICE_PER_UNIT', 'UNITS_SOLD', 'TOTAL_SALES']].head()

,PRICE_PER_UNIT,UNITS_SOLD,TOTAL_SALES
0,50.0,1200,60000.0
1,50.0,1250,62500.0
2,50.0,1220,61000.0
3,50.0,1200,60000.0
4,60.0,1220,73200.0


In [9]:
#which product had the highest sales in 2021? How much did it sell?
# Product-level question: use the full dataframe, no exclusions needed
product_sales_2021 = (df_product_and_sales[df_product_and_sales['YEAR'] == 2021].groupby('PRODUCT_NAME')['TOTAL_SALES'].sum().sort_values(ascending=False))

print(product_sales_2021)


PRODUCT_NAME
Men's Street Footwear        22687827.0
Women's Apparel              19178278.0
Men's Athletic Footwear      16339593.0
Women's Street Footwear      13535783.0
Men's Apparel                13025293.0
Women's Athletic Footwear    11185004.0
Name: TOTAL_SALES, dtype: float64


In [10]:
#what state had the highest sales of womens products in 2021?
# State-level question: use df_state
womens = df_state[(df_state['YEAR'] == 2021) &
                  (df_state['PRODUCT_NAME'].str.startswith("Women's"))]

womens_agg = womens.groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending=False)

print(womens_agg.head(10))

STATE
Maine            2176301.0
Delaware         2023575.0
New Hampshire    1916400.0
Arizona          1798900.0
Missouri         1771992.0
Virginia         1744261.0
Illinois         1743277.0
Nebraska         1712680.0
Connecticut      1600156.0
New York         1505256.0
Name: TOTAL_SALES, dtype: float64


In [11]:
#which state had the highest sales of mens products in 2021?
#what state had the highest sales of womens products in 2021?
# State-level question: use df_state
mens = df_state[(df_state['YEAR'] == 2021) &
                  (df_state['PRODUCT_NAME'].str.startswith("Men's"))]

mens_agg = mens.groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending=False)

print(mens_agg.head(10))

STATE
Delaware         2334300.0
Arizona          2261025.0
New Hampshire    2232000.0
Maine            2217190.0
Illinois         2093438.0
Missouri         1951530.0
Connecticut      1926568.0
New York         1847069.0
Nebraska         1718726.0
New Mexico       1708196.0
Name: TOTAL_SALES, dtype: float64


In [12]:
#which retailer purchased the most units in 2021? 2020?
# Retailer-level question: use df_retailer (excludes the three W00xxxx IDs)
for year in [2021, 2020]:
    units_by_year = (df_retailer[df_retailer['YEAR'] == year].groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False))
    print(f"\n=== {year} ===")
    print(units_by_year)



=== 2021 ===
RETAILER
Foot Locker      1098160
West Gear         269714
Sports Direct     248296
Amazon            205570
Kohl's            136950
Walmart            12270
Name: UNITS_SOLD, dtype: int64

=== 2020 ===
RETAILER
Amazon           317930
Kohl's            68686
West Gear         57334
Sports Direct     18399
Name: UNITS_SOLD, dtype: int64


In [13]:
# Year-over-year dollar sales (product frame, no exclusions needed)
yearly = df_product_and_sales.groupby('YEAR')['TOTAL_SALES'].sum()

print(yearly.apply(lambda x: f"${x:,.0f}"))
print(f"\nGrowth: ${yearly[2021] - yearly[2020]:,.0f}")

YEAR
2020    $24,221,575
2021    $95,951,778
Name: TOTAL_SALES, dtype: object

Growth: $71,730,203


In [14]:
#How have our number of locations grown?
# Exclude the placheodler null ID — it isn't a real location
active = df_product_and_sales[df_product_and_sales['RETAILER_ID'] != '999999999']

r2020 = set(active[active['YEAR'] == 2020]['RETAILER_ID'])
r2021 = set(active[active['YEAR'] == 2021]['RETAILER_ID'])

print(f"locations in 2020: {len(r2020)}")
print(f"locations in 2021: {len(r2021)}")
print(f"  Retained: {len(r2020 & r2021)}")
print(f"  New in 2021: {len(r2021 - r2020)}")
print(f"  Churned after 2020: {len(r2020 - r2021)}")

locations in 2020: 25
locations in 2021: 100
  Retained: 19
  New in 2021: 81
  Churned after 2020: 6


In [15]:
#did the locations we retained grow their sales?
# Split 2021 dollars by whether the location was already selling in 2020
sales_2021 = active[active['YEAR'] == 2021]
existing = sales_2021[sales_2021['RETAILER_ID'].isin(r2020)]['TOTAL_SALES'].sum()
new = sales_2021[~sales_2021['RETAILER_ID'].isin(r2020)]['TOTAL_SALES'].sum()
base_2020 = active[active['YEAR'] == 2020]['TOTAL_SALES'].sum()

total_growth = (existing + new) - base_2020

print(f"2020 baseline:              ${base_2020:,.0f}")
print(f"2021 from existing:         ${existing:,.0f}")
print(f"2021 from new locations:    ${new:,.0f}")
#Sales driven by new locations. We had churn of locations. how did the ones we kept increase/decrease?

2020 baseline:              $24,221,575
2021 from existing:         $16,221,883
2021 from new locations:    $79,712,015


In [16]:
# locations present in both years
retained = set(active[active['YEAR'] == 2020]['RETAILER_ID']) & \
           set(active[active['YEAR'] == 2021]['RETAILER_ID'])

print(f"Locations in both years: {len(retained)}")

# Their sales, grouped by year
same_store = active[active['RETAILER_ID'].isin(retained)]
print(same_store.groupby('YEAR')['TOTAL_SALES'].sum())

Locations in both years: 19
YEAR
2020    12842322.0
2021    16221883.0
Name: TOTAL_SALES, dtype: float64


In [17]:
by_year = df_retailer.pivot_table(
    index='RETAILER', columns='YEAR', values='TOTAL_SALES', aggfunc='sum'
)

print("Share of each year's dollars (%):")
print((by_year / by_year.sum() * 100).round(1))

Share of each year's dollars (%):
YEAR           2020  2021
RETAILER                 
Amazon         72.3  11.1
Foot Locker     NaN  57.0
Kohl's         14.9   7.7
Sports Direct   3.7  12.4
Walmart         NaN   0.5
West Gear       9.0  11.3


In [18]:
# How do our sales break down by channel
mix = df_product_and_sales.pivot_table(
    index='SALES_METHOD',
    columns='YEAR',
    values='TOTAL_SALES',
    aggfunc='sum'
)

print(mix)
print("\nShare of total (%):")
print((mix / mix.sum() * 100).round(1))
#Sales are moving towards online

YEAR                2020        2021
SALES_METHOD                        
In-store       9374550.0  26274075.0
Online         4519966.0  40443856.0
Outlet        10327059.0  29233847.0

Share of total (%):
YEAR          2020  2021
SALES_METHOD            
In-store      38.7  27.4
Online        18.7  42.2
Outlet        42.6  30.5


In [19]:
# Channel mix by region
region_mix = df_state.pivot_table(
    index='REGION',
    columns='SALES_METHOD',
    values='TOTAL_SALES',
    aggfunc='sum'
)

print("Share of each region's dollars (%):")
print((region_mix.div(region_mix.sum(axis=1), axis=0) * 100).round(1))
#different regions have different channel preferences

Share of each region's dollars (%):
SALES_METHOD  In-store  Online  Outlet
REGION                                
Midwest           39.2    35.4    25.4
Northeast         16.6    48.4    35.0
South             21.2    69.9     8.9
Southeast         52.2    20.9    26.9
West              28.5    18.2    53.3


In [20]:
# Channel mix by product category
product_mix = df_product_and_sales.pivot_table(
    index='PRODUCT_NAME',
    columns='SALES_METHOD',
    values='TOTAL_SALES',
    aggfunc='sum'
)

print("Share of each category's dollars (%):")
print((product_mix.div(product_mix.sum(axis=1), axis=0) * 100).round(1))
#doesn't seem to vary

Share of each category's dollars (%):
SALES_METHOD               In-store  Online  Outlet
PRODUCT_NAME                                       
Men's Apparel                  29.9    38.3    31.8
Men's Athletic Footwear        28.8    37.1    34.1
Men's Street Footwear          32.1    37.5    30.5
Women's Apparel                29.4    37.6    33.0
Women's Athletic Footwear      28.3    37.5    34.3
Women's Street Footwear        28.1    36.6    35.2
